# ☕ EDA — Khảo sát phân phối `coffee_articles.csv`

Notebook này thực hiện Exploratory Data Analysis (EDA) toàn diện trên tập báo thu thập được,  
bao gồm: phân phối thời gian, domain, độ dài bài, keyword presence, và noise check.

---


## 0. Imports & Cấu hình

In [3]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# Pandas display
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.1f}".format)

CSV_PATH = r"D:\Git\coffee-price-ie\data\03_articles\coffee_articles.csv"

## 1. Load dữ liệu

Đọc CSV, parse ngày tháng, tính độ dài content.


In [4]:
df = pd.read_csv(CSV_PATH)
df["PUBLISHED_DATE"] = pd.to_datetime(df["PUBLISHED_DATE"], errors="coerce")
df["content_len"] = df["CONTENT"].fillna("").apply(len)
df["year"] = df["PUBLISHED_DATE"].dt.year

print(f"Tổng số rows  : {len(df):,}")
print(f"Columns       : {list(df.columns)}")
print(f"Null CONTENT  : {df['CONTENT'].isna().sum():,}")
print(f"Null DATE     : {df['PUBLISHED_DATE'].isna().sum():,}")
df.head(3)

Tổng số rows  : 9,553
Columns       : ['URL_HASH', 'TARGET', 'QUERY_ID', 'DOMAIN', 'URL', 'PUBLISHED_DATE', 'DISCOVERED_AT', 'CONTENT', 'content_len', 'year']
Null CONTENT  : 0
Null DATE     : 323


,URL_HASH,TARGET,QUERY_ID,DOMAIN,URL,PUBLISHED_DATE,DISCOVERED_AT,CONTENT,content_len,year
0,0378154854331b7a33930dcdbe5464e49fcec7aa1e11ed...,robusta,robusta_general,www.konnaicoffee.vn,https://www.konnaicoffee.vn/konnai-coffee-huon...,2024-06-22,2026-03-19 18:29:49.333,Konnai Coffee: Hương vị Sơn La vang danh châu ...,9622,"2,024.0"
1,4b69332104762862f48aa41bb6790772e1f7060a09e105...,robusta,robusta_general,thoibaotaichinhvietnam.vn,https://thoibaotaichinhvietnam.vn/gia-vang-hom...,2024-06-22,2026-03-19 18:29:48.327,"Giá vàng hôm nay (22/6): Vàng thế giới ""quay x...",22394,"2,024.0"
2,2c1a0eb25e8f183257484a1c9de97b532d58f07cbb735e...,robusta,robusta_general,baoquocte.vn,https://baoquocte.vn/gia-tieu-hom-nay-2262024-...,2024-06-22,2026-03-19 18:29:47.347,"Giá tiêu hôm nay 22/6/2024, xuất khẩu dự kiến ...",15283,"2,024.0"


## 2. Phân phối theo thời gian × Target

Crosstab số bài theo **năm** và **loại cà phê** (robusta / arabica).


In [5]:
pd.crosstab(df["year"], df["TARGET"], margins=True)

TARGET,arabica,robusta,All
year,,,
"2,021.0",2,0,2
"2,022.0",1,0,1
"2,023.0",1121,1159,2280
"2,024.0",1762,1444,3206
"2,025.0",1516,1641,3157
"2,026.0",285,296,581
"2,566.0",0,3,3
All,4687,4543,9230


## 3. Top 20 Domains

Xem nguồn báo nào đóng góp nhiều bài nhất.


In [6]:
domain_counts = df["DOMAIN"].value_counts().head(20).reset_index()
domain_counts.columns = ["domain", "count"]
domain_counts.style.bar(subset=["count"], color="#c8a96e")

,domain,count
0,thoibaotaichinhvietnam.vn,855
1,vietnambiz.vn,624
2,vov.vn,589
3,baoquocte.vn,501
4,laodong.vn,490
5,trangtraiviet.danviet.vn,438
6,nld.com.vn,359
7,kinhtedothi.vn,295
8,cafef.vn,285
9,baonghean.vn,273


## 4. Phân phối độ dài Content

Phân tích thống kê và phân nhóm độ dài bài (tính theo ký tự).


In [7]:
# Thống kê mô tả
desc = df["content_len"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99])
print("Thống kê độ dài content:")
print(desc.round(0))

Thống kê độ dài content:
count     9,553.0
mean     11,021.0
std      11,098.0
min         300.0
10%       5,243.0
25%       7,357.0
50%       9,765.0
75%      13,107.0
90%      17,039.0
95%      20,934.0
99%      27,323.0
max     674,268.0
Name: content_len, dtype: float64


In [8]:
# Phân nhóm
bins   = [0, 200, 500, 1000, 3000, 10_000, 999_999]
labels = ["<200", "200-500", "500-1k", "1k-3k", "3k-10k", ">10k"]
df["len_bucket"] = pd.cut(df["content_len"], bins=bins, labels=labels)

bucket_df = (
    df["len_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)
bucket_df.columns = ["bucket", "count"]
bucket_df["pct"] = (bucket_df["count"] / bucket_df["count"].sum() * 100).round(1)
bucket_df.style.bar(subset=["count"], color="#6aab7b")


,bucket,count,pct
0,<200,0,0.000000
1,200-500,2,0.000000
2,500-1k,2,0.000000
3,1k-3k,59,0.600000
4,3k-10k,4945,51.800000
5,>10k,4545,47.600000


## 5. Sample content ngắn (<300 ký tự) — thường là noise

Những bài cực ngắn thường là quảng cáo, snippet, hoặc lỗi crawl.


In [9]:
short = df[df["content_len"] < 300].sample(min(5, len(df[df["content_len"] < 300])), random_state=42)

for _, row in short.iterrows():
    date_str = row["PUBLISHED_DATE"].date() if pd.notna(row["PUBLISHED_DATE"]) else "N/A"
    print(f"DOMAIN : {row['DOMAIN']}  |  DATE : {date_str}  |  LEN : {row['content_len']}")
    print(f"CONTENT: {repr(str(row['CONTENT'])[:200])}")
    print("-" * 80)


## 6. Sample content dài (>3 000 ký tự) — bài báo thực

Bài có nội dung đầy đủ là nguồn thông tin chính cho pipeline IE.


In [10]:
long_df = df[df["content_len"] > 3000].sample(min(3, len(df[df["content_len"] > 3000])), random_state=42)

for _, row in long_df.iterrows():
    date_str = row["PUBLISHED_DATE"].date() if pd.notna(row["PUBLISHED_DATE"]) else "N/A"
    print(f"DOMAIN : {row['DOMAIN']}  |  DATE : {date_str}  |  LEN : {row['content_len']:,}")
    print(f"CONTENT (500 ký đầu):")
    print(repr(str(row["CONTENT"])[:500]))
    print("-" * 80)

DOMAIN : trangtraiviet.danviet.vn  |  DATE : 2024-05-18  |  LEN : 7,609
CONTENT (500 ký đầu):
'Mận chín đỏ trên Cao nguyên Mộc Châu\n\nTrang chủ\nThời sự\n\nTin tức\nĐời sống\nXã hội\n\nTrang trại xanh\n\nChủ trang trại\nTrang trại đẹp\nVườn Xanh\n\nTiêu dùng thông minh\n\nHàng hóa\nMua sắm\nĐặt hàng\n\nChuyển động Tây Bắc\n\nĐiện Biên Hôm Nay\nLai Châu Ngày Mới\nHòa Bình thi đua yêu nước\nLào Cai thi đua yêu nước\nTuyên Quang thi đua yêu nước\n\nThời sự\n\nTin tức\nĐời sống\nXã hội\n\nTrang trại xanh\n\nChủ trang trại\nTrang trại đẹp\nVườn Xanh\n\nTiêu dùng thông minh\n\nHàng hóa\nMua sắm\nĐặt hàng\n\nChuyển động Tây Bắc\n\nĐiện Biên H'
--------------------------------------------------------------------------------
DOMAIN : thoibaotaichinhvietnam.vn  |  DATE : 2023-06-02  |  LEN : 18,883
CONTENT (500 ký đầu):
'Mỹ có khoản nợ quốc gia cao nhất thế giới ở mức 31,4 nghìn tỷ USD | Thời báo Tài chính Việt Nam\n\nThứ ba 24/03/2026 02:21 | Hotline: 0362656889 | Email: [email protected]\nThờ

## 7. Keyword Presence

Tỉ lệ bài chứa các từ khoá quan trọng: giá, hướng biến động, địa danh, loại hạt, v.v.


In [11]:
keywords = {
    # Price patterns
    "có số + đồng"   : r"\d[\d.,]*\s*(đồng|VND|vnđ)",
    "có số + /kg"    : r"\d[\d.,]*\s*/\s*kg",
    "có số + nghìn"  : r"\d[\d.,]*\s*(nghìn|ngàn)",
    # Direction
    "tăng"           : r"\btăng\b",
    "giảm"           : r"\bgiảm\b",
    "ổn định"        : r"ổn định",
    "đi ngang"       : r"đi ngang",
    "phục hồi"       : r"phục hồi",
    "sụt giảm"       : r"sụt giảm",
    "lao dốc"        : r"lao dốc",
    # Location
    "Đắk Lắk"        : r"[Đđ]ắk [Ll]ắk",
    "Lâm Đồng"       : r"[Ll]âm [Đđ]ồng",
    "Tây Nguyên"     : r"[Tt]ây [Nn]guyên",
    "Gia Lai"        : r"[Gg]ia [Ll]ai",
    # Commodity
    "robusta"        : r"[Rr]obusta",
    "arabica"        : r"[Aa]rabica",
    "cà phê nhân"    : r"cà phê nhân",
    # Price context
    "giá cà phê"     : r"giá cà phê",
    "thị trường"     : r"thị trường",
    "xuất khẩu"      : r"xuất khẩu",
    # Noise indicators
    "mua ngay"       : r"mua ngay",
    "đặt hàng"       : r"đặt hàng",
    "liên hệ"        : r"liên hệ",
    "quảng cáo"      : r"[Qq]uảng cáo",
}

content_series = df["CONTENT"].fillna("")
rob = df[df["TARGET"] == "robusta"]["CONTENT"].fillna("")
ara = df[df["TARGET"] == "arabica"]["CONTENT"].fillna("")

rows = []
for label, pattern in keywords.items():
    rows.append({
        "keyword"    : label,
        "% tổng"     : content_series.str.contains(pattern, regex=True, na=False).mean(),
        "% robusta"  : rob.str.contains(pattern, regex=True, na=False).mean(),
        "% arabica"  : ara.str.contains(pattern, regex=True, na=False).mean(),
    })

kw_df = pd.DataFrame(rows)
kw_df = kw_df.set_index("keyword")

kw_df.style.format("{:.1%}").bar(
    subset=["% tổng"], color="#c8a96e", vmin=0, vmax=1
).background_gradient(subset=["% robusta", "% arabica"], cmap="YlGn")


,% tổng,% robusta,% arabica
keyword,,,
có số + đồng,70.4%,75.1%,65.9%
có số + /kg,0.5%,0.7%,0.4%
có số + nghìn,13.9%,15.0%,12.9%
tăng,89.2%,92.5%,86.1%
giảm,87.2%,91.5%,83.1%
ổn định,49.2%,53.4%,45.0%
đi ngang,24.2%,28.0%,20.4%
phục hồi,34.7%,37.8%,31.7%
sụt giảm,13.2%,14.0%,12.4%


## 8. Sample 1 bài từ mỗi Top-5 Domain

Kiểm tra chất lượng nội dung từ các nguồn đóng góp nhiều nhất.


In [12]:
top_domains = df["DOMAIN"].value_counts().head(5).index.tolist()

for domain in top_domains:
    row = df[df["DOMAIN"] == domain].sample(1, random_state=1).iloc[0]
    date_str = row["PUBLISHED_DATE"].date() if pd.notna(row["PUBLISHED_DATE"]) else "N/A"
    print(f"{'─'*60}")
    print(f"  DOMAIN : {domain}")
    print(f"  DATE   : {date_str}  |  LEN : {row['content_len']:,} ký tự")
    print(f"  TEXT   : {repr(str(row['CONTENT'])[:500])}")
print("─" * 60)


────────────────────────────────────────────────────────────
  DOMAIN : thoibaotaichinhvietnam.vn
  DATE   : 2025-12-30  |  LEN : 9,433 ký tự
  TEXT   : 'Ngày 30/12: Giá cà phê quay đầu giảm mạnh, hồ tiêu ổn định ở mức cao | Thời báo Tài chính Việt Nam\n\nThứ tư 25/03/2026 08:03 Hotline: 0965 199 586 Email: [email protected]\nThời tiết:\nHà Nội 24°C\nĐà Nẵng 23°C\nTP Hồ Chí Minh 27°C\nXem thêm\nVNI: 1,614.77 - 23.60 (1.48%)\nKL: 771,713,376 (CP) GT: 20,461 (tỷ)\n273 48 59 Đóng cửa\nVN30: 1,770.16 - 29.11 (1.67%)\nKL: 319,976,263 (CP) GT: 10,776 (tỷ)\n25 2 3 Đóng cửa\nHNX: 243.81 - 6.27 (2.64%)\nKL: 68,806,901 (CP) GT: 1,188 (tỷ)\n117 34 49 Đóng cửa\nHNX30: 515.93'
────────────────────────────────────────────────────────────
  DOMAIN : vietnambiz.vn
  DATE   : 2025-08-23  |  LEN : 12,645 ký tự
  TEXT   : 'Giá tiêu hôm nay 23/8: Tăng tới 3.000 đồng/kg ở một số địa phương\n\n|\n\nTHỜI SỰ\nDỰ BÁO\nHÀNG HÓA\nQUỐC TẾ\nTÀI CHÍNH\nNHÀ ĐẤT\nCHỨNG KHOÁN\nDOANH NGHIỆP\nKINH DOANH\nDỮ LIỆU\n\nThời

## 9. Số bài / ngày — Phân phối

Xem mật độ thu thập: trung bình mấy bài/ngày, ngày nào spike, ngày nào thiếu.


In [20]:
# Lọc dữ liệu trong khoảng thời gian phân tích chính (11/03/2023 - 11/03/2026)
start_date = pd.to_datetime("2023-03-11")
end_date = pd.to_datetime("2026-03-11")

df_filtered = df[(df["PUBLISHED_DATE"] >= start_date) & (df["PUBLISHED_DATE"] <= end_date)]

daily_counts = df_filtered.groupby(["TARGET", df_filtered["PUBLISHED_DATE"].dt.date]).size()

print(f"Thống kê số bài/ngày theo target (từ {start_date.date()} đến {end_date.date()}):")
display(daily_counts.groupby(level=0).describe().round(1))

print("\nTop 10 ngày nhiều bài nhất:")
display(daily_counts.sort_values(ascending=False).head(10).reset_index(name="count"))

Thống kê số bài/ngày theo target (từ 2023-03-11 đến 2026-03-11):


,count,mean,std,min,25%,50%,75%,max
TARGET,,,,,,,,
arabica,"1,077.0",4.3,2.1,1.0,3.0,4.0,6.0,13.0
robusta,"1,071.0",4.2,2.1,1.0,3.0,4.0,5.0,12.0



Top 10 ngày nhiều bài nhất:


,TARGET,PUBLISHED_DATE,count
0,arabica,2023-11-28,13
1,robusta,2025-07-17,12
2,robusta,2025-05-26,11
3,robusta,2023-12-20,11
4,robusta,2025-07-14,11
5,robusta,2025-08-10,11
6,robusta,2023-10-10,11
7,arabica,2025-07-15,11
8,arabica,2024-09-28,11
9,arabica,2024-10-22,11


In [19]:
# Ngày thiếu dữ liệu trong khoảng thời gian phân tích chính (11/03/2023 - 11/03/2026)
start_date = pd.to_datetime("2023-03-11")
end_date = pd.to_datetime("2026-03-11")

valid_dates = df["PUBLISHED_DATE"].dropna()
valid_dates = valid_dates[(valid_dates >= start_date) & (valid_dates <= end_date)]

date_range = pd.date_range(start_date, end_date)
covered    = valid_dates.dt.date.unique()
missing    = [d.date() for d in date_range if d.date() not in covered]

print(f"Khoảng phân tích: {start_date.date()} đến {end_date.date()}")
print(f"Tổng ngày trong khoảng : {len(date_range)}")
print(f"Ngày có ≥ 1 bài        : {len(covered)}")
print(f"Ngày không có bài      : {len(missing)}")
if missing[:10]:
    print(f"Ví dụ 10 ngày thiếu đầu tiên:\n{missing[:10]}")

Khoảng phân tích: 2023-03-11 đến 2026-03-11
Tổng ngày trong khoảng : 1097
Ngày có ≥ 1 bài        : 1097
Ngày không có bài      : 0


## 10. Noise Check

Bài **không** chứa pattern giá lẫn keyword hướng biến động  
→ nhiều khả năng là noise (quảng cáo, tin không liên quan).


In [21]:
content_series = df["CONTENT"].fillna("")

has_price     = content_series.str.contains(
    r"\d[\d.,]*\s*(đồng|VND|/kg|nghìn|ngàn)", regex=True, na=False
)
has_direction = content_series.str.contains(
    r"\b(tăng|giảm|ổn định|đi ngang|sụt|phục hồi)\b", regex=True, na=False
)
has_either = has_price | has_direction

print(f"Bài có pattern giá       : {has_price.sum():,}  ({has_price.mean():.1%})")
print(f"Bài có kw direction      : {has_direction.sum():,}  ({has_direction.mean():.1%})")
print(f"Bài có ít nhất 1 trong 2 : {has_either.sum():,}  ({has_either.mean():.1%})")
print(f"Bài noise (không có gì)  : {(~has_either).sum():,}  ({(~has_either).mean():.1%})")


Bài có pattern giá       : 7,094  (74.3%)
Bài có kw direction      : 9,047  (94.7%)
Bài có ít nhất 1 trong 2 : 9,143  (95.7%)
Bài noise (không có gì)  : 410  (4.3%)


In [22]:
# Noise theo năm
df["has_signal"] = has_either

noise_by_year = df.groupby("year")["has_signal"].agg(total="count", has_signal="sum")
noise_by_year["pct_signal"] = noise_by_year["has_signal"] / noise_by_year["total"]
noise_by_year["pct_noise"]  = 1 - noise_by_year["pct_signal"]

noise_by_year.style.format({
    "total"      : "{:,}",
    "has_signal" : "{:,}",
    "pct_signal" : "{:.1%}",
    "pct_noise"  : "{:.1%}",
}).bar(subset=["pct_noise"], color="#e07b54")


,total,has_signal,pct_signal,pct_noise
year,,,,
2021.000000,2,2,100.0%,0.0%
2022.000000,1,1,100.0%,0.0%
2023.000000,"2,280","2,077",91.1%,8.9%
2024.000000,"3,206","3,022",94.3%,5.7%
2025.000000,"3,157","3,146",99.7%,0.3%
2026.000000,581,580,99.8%,0.2%
2566.000000,3,3,100.0%,0.0%


---

## ✅ Tóm tắt & Bước tiếp theo

| Hạng mục | Nhận xét |
|---|---|
| **Temporal coverage** | Kiểm tra crosstab ở Section 2 |
| **Domain quality** | Top-domain sample ở Section 8 |
| **Content length** | Bài <200 ký tự nên filter trước IE |
| **Keyword signal** | Xem tỉ lệ `tăng`/`giảm`/`giá cà phê` ở Section 7 |
| **Noise rate** | Bài thiếu cả giá lẫn direction → candidate để loại |

**Gợi ý bước tiếp theo:**
- Filter noise (content_len < 200 hoặc không có signal)  
- Thiết kế prompt IE dựa trên keyword distribution  
- Phân tầng train/test theo năm để tránh data leakage
